# AI-Based Diabetes Prediction System
## Exploratory Data Analysis & Model Training Notebook

**Student:** Nafize Ali  
**Branch:** Artificial Intelligence & Data Science  
**Type:** B.Tech Minor Project  

---

This notebook covers the full end-to-end workflow of the project:
1. Data Loading & Initial Inspection
2. Exploratory Data Analysis (EDA)
3. Data Cleaning & Preprocessing
4. Feature Engineering & Selection
5. Model Training & Comparison
6. Model Evaluation & Interpretation
7. Saving the Final Model

## 1. Import Libraries

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    roc_curve, classification_report
)

warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8fafc',
    'axes.edgecolor': '#cbd5e1',
    'font.family': 'sans-serif',
    'font.size': 10
})

print('Libraries imported successfully.')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

## 2. Load Dataset

In [ ]:
DATASET_PATH = '../dataset/diabetes_binary_health.csv'
df = pd.read_csv(DATASET_PATH)

print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
# Basic dataset information
print('=== Dataset Info ===')
df.info()
print()
print('=== Missing Values (NaN) ===')
print(df.isnull().sum())
print()
print('=== Class Distribution (Target: Outcome) ===')
print(df['Outcome'].value_counts())
print(f'Non-Diabetic: {(df["Outcome"]==0).sum()} ({(df["Outcome"]==0).mean()*100:.1f}%)')
print(f'Diabetic: {(df["Outcome"]==1).sum()} ({(df["Outcome"]==1).mean()*100:.1f}%)')

In [ ]:
# Statistical summary
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class Distribution Bar Chart
fig, ax = plt.subplots(figsize=(5.5, 4))
counts = df['Outcome'].value_counts()
bars = ax.bar(
    ['Non-Diabetic (0)', 'Diabetic (1)'],
    [counts[0], counts[1]],
    color=['#3b82f6', '#ef4444'],
    width=0.5, edgecolor='#94a3b8'
)
for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2.0, h + 2,
        f'{int(h)} ({h/len(df)*100:.1f}%)',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )
ax.set_title('Diabetes Class Distribution', fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel('Patient Count')
ax.set_ylim(0, max(counts) * 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr = df.corr()
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='Blues',
    linewidths=0.5, linecolor='#e2e8f0',
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print('\nTop correlations with Outcome (target):')
print(corr['Outcome'].drop('Outcome').sort_values(ascending=False).to_string())

In [ ]:
# KDE Distributions for key features, stratified by Outcome
key_features = ['Glucose', 'BMI', 'Age', 'BloodPressure']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, col in zip(axes, key_features):
    for outcome, label, color in [(0, 'Non-Diabetic', '#3b82f6'), (1, 'Diabetic', '#ef4444')]:
        group = df[df['Outcome'] == outcome][col]
        # Replace 0s for biological zero columns before plotting
        if col in ['Glucose', 'BMI', 'BloodPressure']:
            group = group.replace(0, np.nan).dropna()
        sns.kdeplot(group, ax=ax, label=label, color=color, fill=True, alpha=0.25)
    ax.set_title(f'{col} Distribution by Outcome', fontsize=10, fontweight='bold')
    ax.set_xlabel(col, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Feature Distributions Stratified by Diabetes Outcome', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Data Cleaning — Biological Zero Treatment

In [ ]:
# Columns where 0 is biologically impossible (unrecorded values)
BIOLOGICAL_ZERO_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print('=== Zero Value Counts (Biological Zeros = Missing Values) ===')
for col in BIOLOGICAL_ZERO_COLS:
    zero_count = (df[col] == 0).sum()
    pct = zero_count / len(df) * 100
    print(f'{col:20s}: {zero_count:3d} zeros ({pct:.1f}% of records)')

# Replace zeros with NaN
df_clean = df.copy()
for col in BIOLOGICAL_ZERO_COLS:
    df_clean[col] = df_clean[col].replace(0, np.nan)

print(f'\nMissing values after biological zero treatment:')
print(df_clean.isnull().sum())

## 5. Feature Selection & Train-Test Split

In [ ]:
# Core features selected for the ML pipeline
# Insulin and SkinThickness excluded due to >70% missing data
CORE_FEATURES = ['Age', 'BMI', 'BloodPressure', 'Glucose']
TARGET = 'Outcome'

X = df_clean[CORE_FEATURES]
y = df_clean[TARGET]

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set    : {X_test.shape[0]} samples')
print(f'\nTraining class distribution:')
print(y_train.value_counts())
print(f'\nTest class distribution:')
print(y_test.value_counts())

## 6. Preprocessing Pipeline (No Data Leakage)

In [ ]:
# Build preprocessing pipeline
# IMPORTANT: Fitted on training data ONLY, then applied to test data
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # Fill NaN with training median
    ('scaler',  StandardScaler())                    # Zero-mean, unit-variance normalisation
])

print('Preprocessing pipeline:')
print(preprocessor)

## 7. Model Training & Evaluation

In [ ]:
# Define the three models to compare
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=4, random_state=42, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=42, class_weight='balanced'
    )
}

results    = []
pipelines  = {}
roc_data   = {}
cm_data    = {}

for name, clf in models.items():
    # Build full pipeline: preprocessor + classifier
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)

    results.append({'Model': name, 'Accuracy': round(acc,4),
                    'Precision': round(prec,4), 'Recall': round(rec,4),
                    'F1-Score': round(f1,4), 'ROC-AUC': round(auc,4)})

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, auc)
    cm_data[name]  = confusion_matrix(y_test, y_pred)

df_results = pd.DataFrame(results)
print('=== MODEL PERFORMANCE COMPARISON (TEST SET) ===')
print(df_results.to_string(index=False))

## 8. Visualising Results

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, cm) in zip(axes, cm_data.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Pred 0', 'Pred 1'], yticklabels=['Actual 0', 'Actual 1'])
    ax.set_title(f'{name}\nConfusion Matrix', fontsize=10, fontweight='bold')
plt.suptitle('Confusion Matrices — Test Set', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#2563eb', '#10b981', '#d97706']
for (name, (fpr, tpr, auc)), color in zip(roc_data.items(), colors):
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random Classifier (AUC = 0.500)')
ax.set_title('ROC Curves — Comparison of All Models', fontsize=12, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Random Forest)
rf = pipelines['Random Forest'].named_steps['classifier']
importances = rf.feature_importances_

df_imp = pd.DataFrame({
    'Feature': CORE_FEATURES,
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(df_imp['Feature'], df_imp['Importance'],
               color='#0284c7', edgecolor='#cbd5e1', height=0.5)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.005, bar.get_y() + bar.get_height()/2.0,
            f'{w:.3f}', va='center', fontsize=9, fontweight='bold')
ax.set_title('Random Forest — Feature Importance (Gini Impurity)', fontsize=12, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_xlim(0, max(importances) * 1.3)
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(df_imp.sort_values('Importance', ascending=False).to_string(index=False))

In [ ]:
# Detailed classification report for the winning model
print('=== RANDOM FOREST — DETAILED CLASSIFICATION REPORT (TEST SET) ===')
y_pred_rf = pipelines['Random Forest'].predict(X_test)
print(classification_report(y_test, y_pred_rf, target_names=['Non-Diabetic', 'Diabetic']))

## 9. Performance Metrics Bar Chart

In [ ]:
# Side-by-side metric comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x      = np.arange(len(metrics))
width  = 0.25
colors = ['#2563eb', '#10b981', '#d97706']

fig, ax = plt.subplots(figsize=(11, 5))
for i, (_, row) in enumerate(df_results.iterrows()):
    vals = [row[m] for m in metrics]
    offset = (i - 1) * width
    rects = ax.bar(x + offset, vals, width, label=row['Model'],
                   color=colors[i], edgecolor='#94a3b8', alpha=0.88)

ax.set_title('Model Performance Comparison (Test Set)', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 10. Save the Final Model

In [ ]:
# The winning model is Random Forest
winning_name = 'Random Forest'
best_pipe    = pipelines[winning_name]
best_metrics = df_results[df_results['Model'] == winning_name].iloc[0].to_dict()

save_payload = {
    'model':           best_pipe,
    'model_name':      winning_name,
    'feature_names':   CORE_FEATURES,
    'all_metrics':     df_results,
    'best_metrics':    best_metrics,
    'total_records':   len(df),
    'author':          'Nafize Ali',
    'department':      'B.Tech AI & Data Science'
}

model_path = '../model/diabetes_model.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)
joblib.dump(save_payload, model_path)
print(f'Model saved to: {model_path}')
print(f'Best model    : {winning_name}')
print(f'Test Accuracy : {best_metrics["Accuracy"]*100:.2f}%')
print(f'ROC-AUC       : {best_metrics["ROC-AUC"]}')

## 11. Summary

| Step | What was done |
|------|---------------|
| Data Loading | Loaded the Pima Indians Diabetes dataset (284 records, 8 features) |
| EDA | Visualised class distribution, correlations, and feature distributions |
| Data Cleaning | Identified and replaced biological zeros with `NaN` |
| Feature Selection | Selected 4 core features: Age, BMI, BloodPressure, Glucose |
| Train-Test Split | 80/20 stratified split — **no leakage** |
| Preprocessing | Median imputation + StandardScaler inside a Pipeline |
| Model Training | Trained Logistic Regression, Decision Tree, Random Forest |
| Evaluation | Compared on Accuracy, Precision, Recall, F1-Score, ROC-AUC |
| Best Model | **Random Forest** — highest accuracy (71.93%) and ROC-AUC (0.7817) |
| Deployment | Model saved with `joblib` and served via Streamlit web app |

---
*This notebook is part of the AI-Based Diabetes Prediction System minor project.*  
*Student: Nafize Ali | B.Tech Artificial Intelligence & Data Science*